# Day 2 Homework - Ollama Webpage Summarizer

Upgrade the Day 1 website summarizer to use a local open-source model via Ollama.


In [ ]:
# Imports
import os
from dotenv import load_dotenv
from scraper import fetch_website_contents
from IPython.display import Markdown, display
from openai import OpenAI
import requests


In [ ]:
# Load env vars if you want to reuse any config
load_dotenv(override=True)


## Connect to Ollama

Make sure Ollama is running locally before executing the cells below.


In [ ]:
# Quick health check (expects Ollama at localhost:11434)
requests.get("http://localhost:11434").status_code


In [ ]:
# Pull a model if you don't have one yet
!ollama pull llama3.2


In [ ]:
OLLAMA_BASE_URL = "http://localhost:11434/v1"
ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key="ollama")


## Prompts and helpers


In [ ]:
system_prompt = """
You are a snarky assistant that analyzes the contents of a website,
and provides a short, snarky, humorous summary, ignoring text that might be navigation related.
Respond in markdown. Do not wrap the markdown in a code block - respond just with the markdown.
"""

user_prompt_prefix = """
Here are the contents of a website.
Provide a short summary of this website.
If it includes news or announcements, then summarize these too.

"""


In [ ]:
def messages_for(website_text):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_prefix + website_text},
    ]


In [ ]:
def summarize(url, model="llama3.2"):
    website_text = fetch_website_contents(url)
    response = ollama.chat.completions.create(
        model=model,
        messages=messages_for(website_text),
    )
    return response.choices[0].message.content


In [ ]:
def display_summary(url, model="llama3.2"):
    summary = summarize(url, model=model)
    display(Markdown(summary))


In [ ]:
display_summary("https://edwarddonner.com")
